# Notebook da Camada Bronze

Notebook (2/5) da Tarefa 2 — Projeto CineData Analytics

Este notebook faz a ingestão dos dados brutos. Ele lê os cinco CSVs do volume `landing` e grava uma tabela Delta para cada um no schema `bronze`. Também extrai a cotação do dólar (PTAX) da API do Banco Central para o período informado nos widgets do notebook (por padrão, os últimos 7 dias) e grava em `bronze.tb_cotacao_dolar`.

## Decisões adotadas
- **Dado cru, sem alteração:** todas as colunas são lidas como texto, sem `inferSchema`. Valores como `"Unknown"` no meio de colunas numéricas são preservados como vieram, e a tipagem e a limpeza ficam para a Silver.
- **Leitura robusta de CSV:** as opções `multiLine` e `escape` tratam textos longos (sinopses, comentários) com vírgulas, aspas e quebras de linha internas, que de outra forma quebrariam as linhas.
- **Rastreabilidade:** cada tabela recebe a coluna `ingestion_datetime`, com o momento exato da carga.
- **Modo `append`:** cada execução acrescenta registros sem apagar as cargas anteriores. Por isso, reexecuções geram duplicatas por design. A deduplicação acontece na Silver, que usa o `ingestion_datetime` para manter a versão mais recente.
- **Padrão de nome de arquivo:** a leitura usa curinga (`*movies_info*.csv`) porque os nomes dos arquivos têm prefixos numéricos e sufixos inconsistentes (`TMDB_IMDB` / `IMDB_TMDB`).
- **API do Banco Central:**
  - As datas vêm de widgets no formato `MM-DD-AAAA`, exigido pela API, e são validadas antes da chamada.
  - O padrão de 7 dias corridos garante que haja dias úteis no período, já que a API não retorna cotação em fins de semana e feriados.
  - `raise_for_status()` interrompe a execução se a API responder com erro HTTP.
  - A chamada tem timeout de 30 segundos. Se a API falhar (fora do ar, timeout, erro HTTP ou resposta inválida), o notebook usa um mock de contingência com 7 cotações terminando na data de execução, com valores baseados na série observada em outubro de 2026, e o Job segue em vez de falhar. A coluna `origem` (`api` ou `mock`) identifica de onde veio cada carga. O mock usa o horário 00:00, então uma cotação real do mesmo dia sempre tem prioridade na Silver.

In [0]:
#parametros do ambiente

from pyspark.sql import functions as F
from datetime import date, timedelta, datetime
import requests

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "cinedata_medallion"
bronze_schema = f"{catalog}.bronze"
volume_path = f"/Volumes/{catalog}/bronze/landing"


# Landing -> Bronze: coluna genres do title.basics (IMDb), do jeito que vem na origem
# Fonte: https://developer.imdb.com/non-commercial-datasets (uso pessoal e não comercial)
caminho_title_basics = f"{volume_path}/title.basics.tsv"
tabela_generos = f"{bronze_schema}.genres_imdb"

# Tabela de cotacao do dólar
tabela_cotacao = f"{catalog}.bronze.tb_cotacao_dolar"

# Tabelas Bronze a partir dos CSVs
tabelas = ["movies_info", "movies_financials", "movies_metrics",
           "credits_and_tags", "movies_reviews"]

#variavel do dia atual 
hoje = date.today()


In [0]:
#Este comando foi adicionado para limpar os widgets do notebook e verificar o funcionamento deles
#dbutils.widgets.removeAll()

In [0]:
# Esta celula serve para definição dos parametros de data do notebook (usado na extração da API)
# Eles podem ser modificados diretamente no cabeçalho do notebook

# definindo as datas de hoje e de 7 dias atrás
#print(f"Data de hoje: {hoje}")
padrao_inicio = (hoje - timedelta(days=7)).strftime("%m-%d-%Y")
padrao_fim = hoje.strftime("%m-%d-%Y")

# definindo os widgets
dbutils.widgets.text("data_inicio", padrao_inicio, "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", padrao_fim, "Data fim (MM-DD-AAAA)")


# datas vindas dos widgets (formato MM-DD-AAAA)
data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")


# imprime os widgets (na atividade 2 estes prints estavam acima da definição dos widgets, o que funcionava se os widgets ja estavam definidos, mas gerava erro na primeira execução)
# Após setados os widgets não mudam em outras execuções, mas dentro de cada job o widget é atualizado
# print("Definindo widgets:")
# print(f"Data início: {data_inicio_formatada}")
# print(f"Data fim: {data_fim_formatada}")

## Criação das tabelas no schema Bronze
Esta célula roda um for, de tabela a tabela carregando os dados contidos nos csvs

In [0]:
# Loop de criação das tabelas
for tabela in tabelas:
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape", '"')
          .csv(f"{volume_path}/*{tabela}*.csv")
          .withColumn("ingestion_datetime", F.current_timestamp()))


    (df.write     # escreve a tabela como delta em modo append
       .format("delta")
       .mode("append")
       .saveAsTable(f"{bronze_schema}.{tabela}"))

    #print(f"{bronze_schema}.{tabela}: {df.count()} linhas lidas")

## Extração e Criação da tabela de Cotação
Esta célula faz a requisição para a API do Banco Central com as cotações, armazena em um df se obtiver resposta e carrega na tabela designada

In [0]:
# TRUNCATE TABLE caso desejado
#spark.sql(f"TRUNCATE TABLE {bronze_schema}.tb_cotacao_dolar")
#print(f"Tabela {bronze_schema}.tb_cotacao_dolar truncada com sucesso.")

In [0]:
# Mock de contingência: 7 cotações terminando em `hoje`, no mesmo formato da API.
# Os valores reproduzem a série observada de 09/10 a 15/10/2026 (incluindo o forward fill dos dias sem cotação) e são aplicados aos últimos 7 dias.
# a coluna "origem" = "mock" identifica essas cargas na Bronze.
# Horário 00:00:00: se já existir cotação real do mesmo dia (publicada por volta das 13h), a real vence a deduplicação por dia na Silver.
valores_mock = [4.9886, 4.9886, 4.9886, 4.9886, 5.0090, 5.0435, 5.0515]

COTACOES_MOCK = [
    {
        "dataHoraCotacao": (hoje - timedelta(days=len(valores_mock) - 1 - i)).strftime("%Y-%m-%d 00:00:00.000"),
        "cotacaoCompra": valor,
    }
    for i, valor in enumerate(valores_mock)
]


In [0]:
# valida o formato antes de chamar a API (erro aqui = data digitada errada)
for d in (data_inicio_formatada, data_fim_formatada):
    datetime.strptime(d, "%m-%d-%Y")

# endpoint PTAX do Banco Central: cotações do período, só data/hora e valor de compra
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'"
    f"&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

# Se a API falhar (fora do ar, timeout, erro HTTP ou resposta inválida),
# o pipeline segue com o mock, em vez de interromper o Job
try:
    resposta = requests.get(url, timeout=30)
    resposta.raise_for_status()
    cotacoes = resposta.json()["value"]
    origem = "api"
except (requests.RequestException, ValueError, KeyError) as erro:
    print(f"AVISO: API do Banco Central indisponível ({erro}). Usando o mock de cotações.")
    cotacoes = COTACOES_MOCK
    origem = "mock"

print(f"Período: {data_inicio_formatada} a {data_fim_formatada} | "
      f"{len(cotacoes)} cotações | origem: {origem}")

# grava só se houver cotações (período só com fim de semana/feriado volta vazio)
if cotacoes:
    df = (spark.createDataFrame(cotacoes)
          .select("dataHoraCotacao", "cotacaoCompra")
          .withColumn("origem", F.lit(origem))                       # rastreia de onde veio cada carga
          .withColumn("ingestion_datetime", F.current_timestamp()))

    # bronze em append; mergeSchema porque a coluna "origem" não existia nas cargas anteriores
    (df.write
       .format("delta")
       .mode("append")
       .option("mergeSchema", "true")
       .saveAsTable(tabela_cotacao))

    display(spark.table(tabela_cotacao).orderBy(F.desc("ingestion_datetime")))
else:
    print("Nenhuma cotação no período (fim de semana/feriado?). Nada gravado.")

In [0]:

# desalinham as colunas e o genres de algumas linhas viria de outra coluna
df_genres_imdb = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("quote", "")
    .csv(caminho_title_basics)
    .select("genres")                                         # sem tratamento: \N e listas com vírgula ficam como vieram
    .withColumn("ingestion_datetime", F.current_timestamp())
)

# Append, seguindo o padrão das demais tabelas Bronze (histórico de cargas)
(df_genres_imdb.write
    .format("delta")
    .mode("append")
    .saveAsTable(tabela_generos))

# Uma linha por carga: mostra quantos registros entraram em cada execução
# display(
#     spark.table(tabela_generos)
#     .orderBy(F.col("ingestion_datetime").desc())
# )

In [0]:
#numero de rows em cada tabela
# tabelas = spark.catalog.listTables(bronze_schema)
# for t in tabelas:
#   print(f"{t.name}: {spark.table(f'{bronze_schema}.{t.name}').count()} linhas")
  

In [0]:
# Celula adicional mostrando todas as tabelas da bronze carregadas (serviu de apoio para as transformações feitas na camada silver)

#for t in tabelas:
#    display(spark.table(f"{bronze_schema}.{t.name}"))                   